Tool Integration with LLM

In [532]:
#imports
import json
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,END
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage,ToolMessage
from typing import List,TypedDict,Annotated
from langchain_core.tools import tool

In [533]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_LLM_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_LLM_API_KEY")

In [534]:
# Setting Up Logger
import sys

# Get the path to the parent directory
parent_dir = os.path.abspath("..")

# Add it to the search path if it's not already there
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
    
from RSYLogger.Logger import Logger
logger = Logger("3 Tools Integration with LLM")

In [535]:
#Defining Tool

@tool
def getOrderItems(orderId:str)->str:
    """
    getOrderItems: Function to fetch the list of items as string with respect to order id
    arguments:
    orderId (str) : order id of the order
    returns 
    string of itemIds present in the order

    if order doesnot exist returns -1
    """
    logger.info("Call to getOrderItems")
    logger.info(f"orderId : {orderId} ")

    if orders.get(orderId):
        res = json.dumps(orders.get(orderId))
    else:
        res = "-1"
    
    logger.info(f"Output from getOrderItems : {res}")
    return res

#Test: (remove @tool as well)
# print(getOrderItems("ORD003"))

@tool
def getItemSpecifications(itemId:str)->str:
    """
    getItegetItemSpecifications : Function to fetch the specification of the Item 
    arguments:
    itemId (str) : item id 
    returns 
    string containing the specification of the item 
    name , category , price and return_days

    if items doesnot exist returns -1
    """
    logger.info("Call to getItemSpecifications")
    logger.info(f"itemId : {itemId} ")

    if items.get(itemId):
        res = json.dumps(items.get(itemId))
    else:
        res =  "-1"
    logger.info(f"Output from getItemSpecifications : {res}")
    return res

In [536]:
#initializing LLM
logger.info("LLM initialization : Started")
try:
    llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7,api_key=GROQ_API_KEY)
    # llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7,google_api_key=GEMINI_API_KEY)
    # llm = ChatGoogleGenerativeAI(model="gemini-flash-latest",google_api_key=GEMINI_API_KEY)
    logger.info("LLM initialization : Completed")
except Exception as e:
    logger.info("LLM initialization : Failed")
    logger.info(e)

In [537]:
# 1. Order dictionary: order_id -> list of item_ids
orders = {
    "ORD001": ["ITEM001", "ITEM002"],
    "ORD002": ["ITEM002", "ITEM003"],
    "ORD003": ["ITEM001", "ITEM003", "ITEM004"]
}

# 2. Item dictionary: item_id -> specifications + return days
items = {
    "ITEM001": {
        "name": "Wireless Mouse",
        "category": "Electronics",
        "price": 799,
        "return_days": 7
    },
    "ITEM002": {
        "name": "Bluetooth Headphones",
        "category": "Electronics",
        "price": 1999,
        "return_days": 10
    },
    "ITEM003": {
        "name": "Running Shoes",
        "category": "Footwear",
        "price": 2499,
        "return_days": 15
    },
    "ITEM004": {
        "name": "Backpack",
        "category": "Accessories",
        "price": 1299,
        "return_days": 5
    }
}

In [538]:
tool_list = [getOrderItems, getItemSpecifications]

tools = {t.name: t for t in tool_list}  # dynamic, not hardcoded

llm_with_tools = llm.bind_tools(tool_list)

In [539]:
class GraphState(TypedDict):
    messages:Annotated[List[BaseMessage],lambda x,y:x+y]

In [540]:
def call_llm(state:GraphState):
    logger.info("LLM Called")
    logger.info(f"Current State : {state}")
    messages = state.get("messages")
    systemPrompt = SystemMessage(content="""
You are an Order Manager Chatbot.

IMPORTANT RULES:
- You MUST use tools to fetch order or item data.
- Do NOT guess or make up any data.
- If user asks about an order, ALWAYS call getOrderItems first.
- If item details are needed, call getItemSpecifications.

Available tools:
- getOrderItems(orderId)
- getItemSpecifications(itemId)
""")
    messages = [systemPrompt] + messages
    logger.info(f'System Prompt Added : {messages}')
    response = llm_with_tools.invoke(messages)
    logger.info(f"Response from LLM : {response}")
    return {"messages":[response]}


In [541]:
def shouldCallTools(state:GraphState):
    logger.info("should call tools : conditional check invoked")
    logger.info(f"Current State {state}")
    last = state["messages"][-1]
    logger.info(f"Last Message {last}")
    logger.info(f"Tool Calls {last.tool_calls}")
    if last.tool_calls and len(last.tool_calls):
        logger.info("Tool calls found")
        return True
    else:
        logger.info("Tool calls not found")
        return False

In [542]:
def call_tools(state:GraphState):
    logger.info("calling Tools : call_tools")
    logger.info(f"Current State : {state}")
    messages = state.get("messages")
    last = messages[-1]
    tool_messages = []
    if last.tool_calls:
        logger.info(f"Tool Calls {last.tool_calls}")
        for tool in last.tool_calls:
            logger.info(f"{tool["name"]} : {tool["args"]}")
            toolFn = tools[tool["name"]]
            output = toolFn.invoke(tool["args"])
            tool_messages.append(ToolMessage(content=str(output),tool_call_id=tool["id"]))
            # messages = [ToolMessage(content=str(output),tool_call_id=tool["id"])] + messages
    logger.info(f"Tools output added : {tool_messages}")
    return {"messages":tool_messages}

In [543]:
#workflow

workflow = StateGraph(GraphState)
workflow.add_node("call_llm",call_llm)
workflow.add_node("call_tools",call_tools)

workflow.set_entry_point("call_llm")
workflow.add_conditional_edges("call_llm",shouldCallTools,{
    True:"call_tools",
    False:END
})
workflow.add_edge("call_tools","call_llm")

app = workflow.compile()



In [544]:
def process(prompt):
    response = app.invoke({
    "messages":[HumanMessage(content=prompt)]
    })
    logger.info("Response:\n")
    for message in response["messages"]:
        match message.type:
            case 'ai':
                print(f"AI : {message.content}")
                logger.info(f"AI : {message.content}")
            case 'human':
                print(f"Human : {message.content}")
                logger.info(f"Human : {message.content}")
            case 'tool':
                print(f"Tool Output: {message.content}")
                logger.info(f"Tool Output: {message.content}")

In [545]:
process("What are the Items bought in order ORD003?")

Human : What are the Items bought in order ORD003?
AI : 
Tool Output: ["ITEM001", "ITEM003", "ITEM004"]
AI : The items bought in order ORD003 are ITEM001, ITEM003, and ITEM004.


In [546]:
process("What are the return dates of my Items in order ORD003 ?")

Human : What are the return dates of my Items in order ORD003 ?
AI : 
Tool Output: ["ITEM001", "ITEM003", "ITEM004"]
AI : 
Tool Output: {"name": "Wireless Mouse", "category": "Electronics", "price": 799, "return_days": 7}
Tool Output: {"name": "Running Shoes", "category": "Footwear", "price": 2499, "return_days": 15}
Tool Output: {"name": "Backpack", "category": "Accessories", "price": 1299, "return_days": 5}
AI : The return dates of your items in order ORD003 are:
- ITEM001 (Wireless Mouse): 7 days
- ITEM003 (Running Shoes): 15 days
- ITEM004 (Backpack): 5 days


In [547]:
process("What are the return dates of my order ORD002 ?")

Human : What are the return dates of my order ORD002 ?
AI : 
Tool Output: ["ITEM002", "ITEM003"]
AI : 
Tool Output: {"name": "Bluetooth Headphones", "category": "Electronics", "price": 1999, "return_days": 10}
Tool Output: {"name": "Running Shoes", "category": "Footwear", "price": 2499, "return_days": 15}
AI : The return dates for your order ORD002 are 10 days for the Bluetooth Headphones (ITEM002) and 15 days for the Running Shoes (ITEM003).
